<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.2: FIRRTL AST 遍历

**上一节: [FIRRTL 介绍](4.1_firrtl_ast.ipynb)**<br>
**下一节: [常见传递模式](4.3_firrtl_common_idioms.ipynb)**

### 理解 IR 节点子节点

Writing a Firrtl pass usually requires writing functions which walk the Firrtl datastructure to either collect information or replace IR nodes with new IR nodes.

The IR datastructure is a tree, where each IR node can have some number of children nodes (which in turn can have more children nodes, etc.). IR nodes without children are called leaves.

Different IR nodes can have different children types. 以下 table shows the possible children 类型 for each IR node 类型:

```
+------------+-----------------------------+
|    Node    |          Children           |
+------------+-----------------------------+
| 电路    | DefModule                   |
| DefModule  | Port, Statement             |
| Port       | 类型, Direction             |
| Statement  | Statement, Expression, 类型 |
| Expression | Expression, 类型            |
| 类型       | 类型, Width                 |
| Width      |                             |
| Direction  |                             |
+------------+-----------------------------+
```

### The map 函数

To write a 函数 that traverses a `电路`, we need to first understand the functional programming concept `map`.

#### Understanding Seq.map
A Scala sequence of strings, can be represented as a tree with a root node `Seq` and children nodes `"a"`, `"b"`, and `"c"`:
```scala
val s = Seq("a", "b", "c")
```
```
    Seq
 /   |   \
"a" "b" "c"
```

Suppose we define a 函数 `f` that, given a String 参数 `x`, concatenates `x` with itself:
```scala
def f(x: String): String = x + x
```

We can call `s.map` to return a new `Seq[String]` whose children are the result of applying `f` to every child of s:
```scala
val s = Seq("a", "b", "c")
def f(x: String): String = x + x  // repeated declaration for clarity
val t = s.map(f)
println(t) // Seq("aa", "bb", "cc")
```
```
     Seq
 /    |    \
"aa" "bb" "cc"
```

#### Understanding Firrtl's map

We use this "mapping" idea to create our own, custom `map` methods on IR nodes. Suppose we have a `DoPrim` expression representing 1 + 1; this can be depicted as a tree of expressions with a root node `DoPrim`:
```
        DoPrim
     /          \
UIntValue    UIntValue
```

If we have a 函数 `f` that takes an `Expression` 参数 and returns a new `Expression`, we can "map" it onto all children `Expression` of a given IR node, like our `DoPrim`. This would return 以下 new `DoPrim`, whose children are the result of applying `f` to every `Expression` child of `DoPrim`:
```
        DoPrim
     /          \
f(UIntValue)    f(UIntValue)
```

Sometimes IR nodes have children of multiple types. 例如, `Conditionally` has both `Expression` and `Statement` children. In this case, the map will only apply its 函数 to the children whose 类型 matches the 函数's 参数 类型 (and return 值 类型):
```scala
val c = Conditionally(info, e, s1, s2) // e: Expression, s1, s2: Statement, info: FileInfo
def fExp(e: Expression): Expression = ...
def fStmt(s: Statement): Statement = ...
c.map(fExp)  // Conditionally(fExp(e), s1, s2)
c.map(fStmt) // Conditionally(e, fStmt(s1), fStmt(s2))
```

Scala has "infix notation", which allows you to drop the `.` and parenthesis when calling a 函数 which has one 参数. Often, we write these map functions with infix notation:
```scala
c map fExp  // equivalent to c.map(fExp)
c map fStmt // equivalent to c.map(fStmt)
```

### Pre-order traversal

To traverse a Firrtl tree, we use `map` to write recursive functions which visit every child of every node we care about.

Suppose we want to collect the names of every 寄存器 declared in the 设计; we know this requires visiting every `Statement`. 然而, some `Statement` nodes can have children `Statement`. 因此, we need to write a 函数 that will both check if its 输入 参数 is a `DefRegister` and, if not, will recursively apply `f` to all `Statement` children of its 输入 参数:

以下 函数, `f`, is similar to our described 函数 yet it takes two arguments: a mutable hashset of 寄存器 names, and a `Statement`. Using 函数 currying, we can pass only the first 参数 to return a new 函数 with the desired 类型 signature (`Statement=>Statement`):

```scala
def f(regNames: mutable.HashSet[String]())(s: Statement): Statement = s match {
  // If 寄存器, add name to regNames
  case r: DefRegister =>
    regNames += r.name
    r // Return 参数 unchanged (ok because DefRegister has no Statement children)
  // If not, apply f(regNames) to all children Statement
  case _ => s map f(regNames) // 请注意 f(regNames) is of 类型 Statement=>Statement
}
```

This pattern is very common in Firrtl, and is called "pre-order traversal" because the recursive 函数 matches on the original IR node before recursively applying to its children nodes.

### Post-order traversal

We can write the previous 示例 in a "post-order traversal" as follows:

```scala
def f(regNames: mutable.HashSet[String]())(s: Statement): Statement = 
  // Not we immediately recurse to the children nodes, then match
  s map f(regName) match {
    // If 寄存器, add name to regNames
    case r: DefRegister =>
      regNames += r.name
      r // Return 参数 unchanged (ok because DefRegister has no Statement children)
    // If not, return s
    case _ => s // 请注意 all Statement children of s have had f(regNames) already applied
  }
```

While the traversal ordering is different between these two 示例, it makes no difference for this use case (and many others). 然而, it is an important tool to keep in your back pocket for when the traversal ordering matters.